<a href="https://colab.research.google.com/github/francisagana36-spec/hospital_data_analytics/blob/main/hospital_Data_analytics_Agana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install any extra libraries not in Colab by default
!pip install fuzzywuzzy python-Levenshtein missingno -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 52.0 MB/s eta 0:00:00


In [2]:
# Install Plotly (only needs to run once)
!pip install plotly -q
print('✅ Plotly installed!')

✅ Plotly installed!


In [3]:
# ── PART 1: Static charts ──────────────────────────────────
import matplotlib.pyplot as plt       # The main plotting library
import matplotlib.gridspec as gridspec
import seaborn as sns                 # Makes Matplotlib charts look nicer

# ── PART 2: Interactive charts ─────────────────────────────
import plotly.express as px           # Easy interactive charts
import plotly.graph_objects as go     # More control over interactive charts
from plotly.subplots import make_subplots  # Combine multiple Plotly charts

# ── Data tools ─────────────────────────────────────────────
import pandas as pd                   # For working with tables of data
import numpy as np                    # For numbers and calculations

import warnings
warnings.filterwarnings('ignore')
# Make Matplotlib charts look clean
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('✅ All libraries imported successfully!')
print('   Matplotlib  →  static charts')
print('   Seaborn     →  beautiful static charts')
print('   Plotly      →  interactive charts')

✅ All libraries imported successfully!
   Matplotlib  →  static charts
   Seaborn     →  beautiful static charts
   Plotly      →  interactive charts


In [4]:

import missingno as msno
import re

from scipy import stats
from fuzzywuzzy import fuzz, process
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import LabelEncoder


pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ All libraries loaded successfully!')

✅ All libraries loaded successfully!


In [5]:
from google.colab import files

print("Please upload: patients.csv, admissions.csv, doctors.csv")
uploaded = files.upload()
print("\nFiles uploaded:", list(uploaded.keys()))


Please upload: patients.csv, admissions.csv, doctors.csv


Saving patients.csv to patients.csv
Saving doctors.csv to doctors.csv
Saving admissions.csv to admissions.csv

Files uploaded: ['patients.csv', 'doctors.csv', 'admissions.csv']


In [6]:
# Load the datasets
patients   = pd.read_csv('patients.csv')
admissions = pd.read_csv('admissions.csv')
doctors    = pd.read_csv('doctors.csv')

print(f"patients   → {patients.shape[0]} rows, {patients.shape[1]} columns")
print(f"admissions → {admissions.shape[0]} rows, {admissions.shape[1]} columns")
print(f"doctors    → {doctors.shape[0]} rows, {doctors.shape[1]} columns")


patients   → 200 rows, 8 columns
admissions → 350 rows, 8 columns
doctors    → 40 rows, 6 columns


In [7]:
print("=== PATIENTS ===")
print(patients.head())
print()
patients.info()
print()
patients.describe( )

=== PATIENTS ===
  patient_id  full_name   age  gender blood_type         region date_of_birth  \
0      P0001  Patient_1  4.00  Female         O+  Greater Accra    2020-04-27   
1      P0002  Patient_2 32.00     NaN         A+          Volta    1992-08-12   
2      P0003  Patient_3 14.00       M         A+       Northern    2010-05-27   
3      P0004  Patient_4 70.00    male        AB-        Eastern    1954-04-08   
4      P0005  Patient_5 -7.00    Male         O+        Ashanti           NaN   

         phone  
0          NaN  
1 245074336.00  
2 246059717.00  
3 243030966.00  
4 241689025.00  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   patient_id     200 non-null    object 
 1   full_name      200 non-null    object 
 2   age            191 non-null    float64
 3   gender         181 non-null    object 
 4   blood_type     181 n

,age,phone
count,191.00,191.00
mean,68.26,245796550.61
std,133.15,2590161.09
min,-10.00,241019311.00
25%,18.00,243612365.50
50%,41.00,245896624.00
75%,70.50,248190780.00
max,907.00,249990305.00


In [8]:
# explore admission
print("\n=== ADMISSIONS ===")
print(admissions.head())
print()
admissions.info()
print()
admissions.describe()
#


=== ADMISSIONS ===
  admission_id patient_id  department      diagnosis  admit_date  \
0       A00001      P0005  Cardiology   Hypertension  2023-08-02   
1       A00002      P0092   Neurology            NaN  2022-02-10   
2       A00003      P0170  Pediatrics         Stroke  2022-04-05   
3       A00004      P0061    Oncology   Hypertension  2023-09-17   
4       A00005      P0022  Cardiology  Heart Failure  2022-05-27   

  discharge_date  treatment_cost_usd insurance_covered  
0     2023-08-26             5822.37                No  
1     2022-02-24             6715.30                no  
2     2022-05-04            12388.32                no  
3     2023-10-14             8997.02                No  
4     2022-06-22             9127.53               YES  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   admission_id        3

,treatment_cost_usd
count,330.00
mean,7420.73
std,4578.03
min,-484.00
25%,3258.89
50%,7451.23
75%,11647.47
max,14994.17


In [9]:
# explore doctors
print("\n=== DOCTORS ===")
print(doctors.head())
print()
doctors.info()
print()
doctors.describe()
#


=== DOCTORS ===
  doctor_id    doctor_name specialisation  department  years_experience  \
0      D001  Dr. Surname_1     Oncologist   Neurology              4.00   
1      D002  Dr. Surname_2  Paediatrician   Emergency             11.00   
2      D003  Dr. Surname_3     Oncologist  Cardiology             20.00   
3      D004  Dr. Surname_4     Oncologist  Cardiology             15.00   
4      D005  Dr. Surname_5    Neurologist  Cardiology             19.00   

   salary_usd  
0    93408.42  
1    82741.50  
2    59034.29  
3    95911.68  
4    87533.94  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   doctor_id         40 non-null     object 
 1   doctor_name       40 non-null     object 
 2   specialisation    40 non-null     object 
 3   department        40 non-null     object 
 4   years_experience  39 non-null     float64
 5  

,years_experience,salary_usd
count,39.00,37.00
mean,14.54,75839.50
std,8.42,24483.36
min,2.00,30161.81
25%,8.00,58756.68
50%,15.00,81555.13
75%,19.50,93408.42
max,30.00,118023.74


issues found---PATIENTS
1. date_of_birth dtype change to datetime
2. phone dtype change to object
3. several columns have missing values
4. age values has negative figure and over 907
5. gender has inconsistency value

issues found ----ADMISSION
1. admit_date dtype change to datetime
2. discharge_date dtype change to datetime
3. several columns have missing values
4. treatment_cost_usd has negative figures
5. insurance_cover has inconsistency values

issues found -----DOCTORS

1. years_experience has missing values
2. salary_usd has missing values

In [ ]:
# HANDLING MISSING VALUES

# find number of missing values and percentage accross the dataset
# Missing Value Summary
def missing_summary(df, name):
    """Returns a detailed missing-value report for a given DataFrame."""
    total    = df.isnull().sum()
    percent  = (total / len(df) * 100).round(2)
    summary  = pd.DataFrame({
        'Missing Count': total,
        'Missing %':     percent
    })
    print(f'=== Missing Value Report ── {name} ===')
    display(summary[summary['Missing Count'] > 0].sort_values('Missing %', ascending=False))
    print('\n')


missing_summary(patients, 'Patients')
missing_summary(admissions, 'Admissions')
missing_summary(doctors, 'Doctors')

=== Missing Value Report ── Patients ===


,Missing Count,Missing %
date_of_birth,32,16.00
gender,19,9.50
blood_type,19,9.50
region,18,9.00
age,9,4.50
phone,9,4.50




=== Missing Value Report ── Admissions ===


,Missing Count,Missing %
diagnosis,41,11.71
insurance_covered,30,8.57
department,23,6.57
treatment_cost_usd,20,5.71
discharge_date,11,3.14




=== Missing Value Report ── Doctors ===


,Missing Count,Missing %
salary_usd,3,7.50
years_experience,1,2.50


In [10]:
# – Standardise gender in patients
print("Before cleaning:")
print(patients['gender'].value_counts(dropna=False))
print()

def clean_string_column(series):
  # Add 'nan' to the placeholders to catch string 'Nan' values
  placeholders={'n/a', 'na','none', 'null', 'missing', 'unknown', '-', 'nan'}
  # Create a dictionary for replacements based on title-cased placeholders,
  # as .str.title() is applied before replacement.
  replacements_to_nan = {p.title():np.nan for p in placeholders}

  return (series
          .astype(str)
          .str.strip()
          .str.title() # Apply title case first
          .replace(replacements_to_nan)) # Then replace the resulting title-cased placeholders

for col in ['gender']:
    patients[col] = clean_string_column(patients[col])


print("\nAfter strip:")
print(patients['gender'].value_counts(dropna=False))


# standardise gender to Male and Female
gender_map={'Male':'Male', 'M':'Male', 'Female':'Female', 'F':'Female'}
patients['gender']=patients['gender'].map(gender_map)
print('\nAfter cleaning:')
print(patients['gender'].value_counts(dropna=False))

Before cleaning:
gender
Male      56
Female    46
male      20
NaN       19
female    18
FEMALE    12
MALE      11
M          9
F          9
Name: count, dtype: int64


After strip:
gender
Male      87
Female    76
NaN       19
M          9
F          9
Name: count, dtype: int64

After cleaning:
gender
Male      96
Female    85
NaN       19
Name: count, dtype: int64


In [11]:
# 2.1 – Standardise insurance_covered
print("Before cleaning:")
print(admissions['insurance_covered'].value_counts(dropna=False))
print()

def clean_string_column(series):
  placeholders={'n/a', 'na','none', 'null', 'missing', 'unknown', '-'}
  # Create a dictionary for replacements based on title-cased placeholders,
  # as .str.title() is applied before replacement.
  replacements_to_nan = {p.title():np.nan for p in placeholders}

  return (series
          .astype(str)
          .str.strip()
          .str.title() # Apply title case first
          .replace(replacements_to_nan)) # Then replace the resulting title-cased placeholders

for col in ['insurance_covered']:
    admissions[col] = clean_string_column(admissions[col])



print("\nAfter cleaning:")
print(admissions['insurance_covered'].value_counts(dropna=False))


Before cleaning:
insurance_covered
No     110
Yes    105
yes     44
no      32
NaN     30
YES     29
Name: count, dtype: int64


After cleaning:
insurance_covered
Yes    178
No     142
Nan     30
Name: count, dtype: int64


In [12]:
#  – Fix negative costs & impute
print(f"Negative costs : {(admissions['treatment_cost_usd'] < 0).sum()}")
print(f"Missing costs  : {admissions['treatment_cost_usd'].isna().sum()}")

# convert all numeric value to safe numeric values
def safe_to_numeric(series):
  return pd.to_numeric(series, errors='coerce')

admissions['treatment_cost_usd'] = safe_to_numeric(admissions['treatment_cost_usd'])

# Replace negative costs with NaN
admissions.loc[admissions['treatment_cost_usd'] < 0, 'treatment_cost_usd'] = np.nan
print(f'cost nulls after impossible value removal)  : {admissions['treatment_cost_usd'].isna().sum()}')


print(f"\nAfter fix — Missing costs: {admissions['treatment_cost_usd'].isna().sum()}")
print(f"Cost range: {admissions['treatment_cost_usd'].min():.2f} – {admissions['treatment_cost_usd'].max():.2f}")

# Mean / Median imputation for numeric
admissions['treatment_cost_usd'].fillna(admissions['treatment_cost_usd'].mean(), inplace=True)
print(f"\nCost nulls after mean imputation: {admissions['treatment_cost_usd'].isnull().sum()}")

Negative costs : 13
Missing costs  : 20
cost nulls after impossible value removal)  : 33

After fix — Missing costs: 33
Cost range: 342.00 – 14994.17

Cost nulls after mean imputation: 0


In [13]:
# fixing the date issues

admissions['admit_date']     = pd.to_datetime(admissions['admit_date'])
admissions['discharge_date'] = pd.to_datetime(admissions['discharge_date'])

# Identify bad rows
bad_dates = admissions['discharge_date'] < admissions['admit_date']
print(f"Rows with discharge before admit: {bad_dates.sum()}")

# fix bad dates
admissions.loc[bad_dates, 'discharge_date'] = np.nan


# Create length_of_stay, admit_year, admit_month
admissions['length_of_stay'] = (admissions['discharge_date'] - admissions['admit_date']).dt.days
admissions['admit_year']     = admissions['admit_date'].dt.year
admissions['admit_month']    = admissions['admit_date'].dt.month
admissions['admit_day']    = admissions['admit_date'].dt.day_name()
admissions['discharge_day']    = admissions['discharge_date'].dt.day_name()


print(admissions[['admit_date','discharge_date','length_of_stay','admit_year','admit_month', 'admit_day', 'discharge_day']].head(8))

Rows with discharge before admit: 25
  admit_date discharge_date  length_of_stay  admit_year  admit_month  \
0 2023-08-02     2023-08-26           24.00        2023            8   
1 2022-02-10     2022-02-24           14.00        2022            2   
2 2022-04-05     2022-05-04           29.00        2022            4   
3 2023-09-17     2023-10-14           27.00        2023            9   
4 2022-05-27     2022-06-22           26.00        2022            5   
5 2023-12-28     2024-01-01            4.00        2023           12   
6 2024-02-05     2024-02-13            8.00        2024            2   
7 2024-09-12            NaT             NaN        2024            9   

   admit_day discharge_day  
0  Wednesday      Saturday  
1   Thursday      Thursday  
2    Tuesday     Wednesday  
3     Sunday      Saturday  
4     Friday     Wednesday  
5   Thursday        Monday  
6     Monday       Tuesday  
7   Thursday           NaN  


In [14]:
# 2.4 – Duplicates
print("Duplicate patient_ids  :", patients.duplicated(subset='patient_id').sum())
print("Duplicate admission_ids:", admissions.duplicated(subset='admission_id').sum())
print("Exact duplicate rows (patients)  :", patients.duplicated().sum())
print("Exact duplicate rows (admissions):", admissions.duplicated().sum())
print("Exact duplicate rows (doctors)   :", doctors.duplicated().sum())

# drop exact duplicate rows where found
patients.drop_duplicates(inplace=True)
admissions.drop_duplicates(inplace=True)
doctors.drop_duplicates(inplace=True)

# Drop duplicate patient_id in patients, keeping the first entry
patients.drop_duplicates(subset='patient_id', inplace=True)

# Drop duplicate admission_id in admissions, keeping the first entry
admissions.drop_duplicates(subset='admission_id', inplace=True)

print("\n--- After dropping duplicates ---")
print("Duplicate patient_ids  :", patients.duplicated(subset='patient_id').sum())
print("Duplicate admission_ids:", admissions.duplicated(subset='admission_id').sum())
print("Exact duplicate rows (patients)  :", patients.duplicated().sum())
print("Exact duplicate rows (admissions):", admissions.duplicated().sum())
print("Exact duplicate rows (doctors)   :", doctors.duplicated().sum())


Duplicate patient_ids  : 0
Duplicate admission_ids: 0
Exact duplicate rows (patients)  : 0
Exact duplicate rows (admissions): 0
Exact duplicate rows (doctors)   : 0

--- After dropping duplicates ---
Duplicate patient_ids  : 0
Duplicate admission_ids: 0
Exact duplicate rows (patients)  : 0
Exact duplicate rows (admissions): 0
Exact duplicate rows (doctors)   : 0


In [15]:
# fixing issues with treatment cost

# 2.5 – IQR outlier detection
Q1 = admissions['treatment_cost_usd'].quantile(0.25)
Q3 = admissions['treatment_cost_usd'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

admissions['cost_outlier'] = (admissions['treatment_cost_usd'] < lower) | (admissions['treatment_cost_usd'] > upper)

n_outliers = admissions['cost_outlier'].sum()
print(f"IQR bounds: ${lower:.2f} — ${upper:.2f}")
print(f"Outliers flagged: {n_outliers}")
print(admissions[admissions['cost_outlier']][['admission_id','treatment_cost_usd']].head())

IQR bounds: $-6814.80 — $22431.88
Outliers flagged: 0
Empty DataFrame
Columns: [admission_id, treatment_cost_usd]
Index: []


In [19]:
# save and download specific files
patients_clean=patients.copy()
admissions_clean=admissions.copy()
doctors_clean=doctors.copy()



In [21]:
patients_clean.to_csv('patients_clean.csv', index=False)
admissions_clean.to_csv('admissions_clean.csv', index=False)
doctors_clean.to_csv('doctors_clean.csv', index=False)

files.download('patients_clean.csv')
files.download('admissions_clean.csv')
files.download('doctors_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
# 3.1 – Merge patients and admissions
hospital_df = pd.merge(admissions, patients, on='patient_id', how='left')

print(f"Shape after merge: {hospital_df.shape}")

# Count unmatched admissions
unmatched = hospital_df['full_name'].isna().sum()
print(f"Admissions with no matching patient: {unmatched}")

hospital_df.head(3)

Shape after merge: (350, 21)
Admissions with no matching patient: 0


,admission_id,patient_id,department,diagnosis,admit_date,discharge_date,treatment_cost_usd,insurance_covered,length_of_stay,admit_year,admit_month,admit_day,discharge_day,cost_outlier,full_name,age,gender,blood_type,region,date_of_birth,phone
0,A00001,P0005,Cardiology,Hypertension,2023-08-02,2023-08-26,5822.37,No,24.00,2023,8,Wednesday,Saturday,False,Patient_5,-7.00,Male,O+,Ashanti,1983-07-11,241689025.00
1,A00002,P0092,Neurology,Appendicitis,2022-02-10,2022-02-24,6715.30,No,14.00,2022,2,Thursday,Thursday,False,Patient_92,57.00,Male,A+,Northern,1967-09-08,241668752.00
2,A00003,P0170,Pediatrics,Stroke,2022-04-05,2022-05-04,12388.32,No,29.00,2022,4,Tuesday,Wednesday,False,Patient_170,65.00,Male,A+,Ashanti,1959-09-02,241941524.00


### 4.5 – Clean and Impute `age` in `hospital_df`

It appears that invalid age values (outside the biologically plausible range of 0-120 years) were not fully addressed in the `hospital_df` after the merge, leading to skewed averages in analyses like the average age by blood type. This step will:

1.  Identify and replace ages outside the 0-120 range with `NaN`.
2.  Impute any remaining `NaN` values in the `age` column with the median age from the `hospital_df`.

In [62]:
# Identify and replace invalid ages (e.g., <0 or >120) with NaN
hospital_df.loc[~hospital_df['age'].between(0, 120), 'age'] = np.nan

# Impute any remaining missing age values with the median of the cleaned 'age' column
hospital_df['age'].fillna(hospital_df['age'].median(), inplace=True)

print(f"Missing 'age' values in hospital_df after cleaning and imputation: {hospital_df['age'].isnull().sum()}")
print(f"Age range in hospital_df after cleaning: {hospital_df['age'].min()} – {hospital_df['age'].max()}")

Missing 'age' values in hospital_df after cleaning and imputation: 0
Age range in hospital_df after cleaning: 1.0 – 90.0


In [34]:
dept_summary = doctors.groupby('department').agg(
    num_doctors       = ('doctor_id', 'nunique'),
    avg_doctor_salary = ('salary_usd', 'mean')
).reset_index()

print(dept_summary)

# Check if columns from dept_summary already exist in hospital_df and drop them
# to prevent MergeError on re-execution
columns_to_drop = ['num_doctors', 'avg_doctor_salary']
for col in columns_to_drop:
    if col in hospital_df.columns:
        hospital_df = hospital_df.drop(columns=col)

# Merge into hospital_df
hospital_df = pd.merge(hospital_df, dept_summary, on='department', how='left')

print(f"\nFinal shape: {hospital_df.shape}")
print(hospital_df[['admission_id','department','num_doctors','avg_doctor_salary']].head(3))

         department  num_doctors  avg_doctor_salary
0        Cardiology            5           81780.05
1         Emergency            7           77571.13
2  General Medicine            6           61862.12
3         Neurology            9           82939.65
4          Oncology            2           99789.43
5       Orthopedics            5           80642.97
6        Pediatrics            6           63067.58

Final shape: (350, 27)
  admission_id  department  num_doctors  avg_doctor_salary
0       A00001  Cardiology            5           81780.05
1       A00002   Neurology            9           82939.65
2       A00003  Pediatrics            6           63067.58


In [39]:
# 4.4 – Merge Verification

print("--- Merge Verification ---")

# Check for nulls in columns added from the patient data (e.g., full_name) and department summary (num_doctors, avg_doctor_salary)
print(f"Missing values in 'full_name' (from patients merge): {hospital_df['full_name'].isnull().sum()}")
print(f"Missing values in 'num_doctors' (from department summary merge): {hospital_df['num_doctors'].isnull().sum()}")
print(f"Missing values in 'avg_doctor_salary' (from department summary merge): {hospital_df['avg_doctor_salary'].isnull().sum()}")

# Display info of the final merged DataFrame to see column types and non-null counts
print("\nhospital_df.info():")
hospital_df.info()

# Display a sample of the merged DataFrame to visually confirm the merge
print("\nhospital_df sample (first 5 rows with key merged columns):")
display(hospital_df[['admission_id', 'patient_id', 'department', 'full_name', 'gender', 'blood_type', 'num_doctors', 'avg_doctor_salary']].head())

--- Merge Verification ---
Missing values in 'full_name' (from patients merge): 0
Missing values in 'num_doctors' (from department summary merge): 0
Missing values in 'avg_doctor_salary' (from department summary merge): 0

hospital_df.info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   admission_id         350 non-null    object        
 1   patient_id           350 non-null    object        
 2   department           350 non-null    object        
 3   diagnosis            350 non-null    object        
 4   admit_date           350 non-null    datetime64[ns]
 5   discharge_date       350 non-null    datetime64[ns]
 6   treatment_cost_usd   350 non-null    float64       
 7   insurance_covered    350 non-null    object        
 8   length_of_stay       350 non-null    float64       
 9   admit_year          

,admission_id,patient_id,department,full_name,gender,blood_type,num_doctors,avg_doctor_salary
0,A00001,P0005,Cardiology,Patient_5,Male,O+,5,81780.05
1,A00002,P0092,Neurology,Patient_92,Male,A+,9,82939.65
2,A00003,P0170,Pediatrics,Patient_170,Male,A+,6,63067.58
3,A00004,P0061,Oncology,Patient_61,Female,O+,2,99789.43
4,A00005,P0022,Cardiology,Patient_22,Male,O+,5,81780.05


# PIVOT TABLES

In [36]:

# a pivot table showing average billing amount by medical condition
pivot_diagnosis_cost = pd.pivot_table(
    hospital_df,
    values  = 'treatment_cost_usd',
    index   = 'diagnosis',
    aggfunc = 'mean'
).round(2)

print(pivot_diagnosis_cost)

               treatment_cost_usd
diagnosis                        
Anaemia                   7235.66
Appendicitis              7914.65
COVID-19                  8183.33
Diabetes                  9021.60
Fracture                  6995.26
Heart Failure             7580.53
Hypertension              7164.31
Malaria                   6932.53
Pneumonia                 7844.25
Stroke                    7136.35
Typhoid                   8976.04


In [57]:
#  a pivot table showing patient count by hospital and gender
pivot_patient_gender = pd.pivot_table(
    hospital_df,
    values  = 'patient_id',
    index   = 'department',
    columns = 'gender',
    aggfunc = 'count'
).round(2)

print(pivot_patient_gender)

gender            Female  Male
department                    
Cardiology            14    28
Emergency             34    47
General Medicine      22    36
Neurology             12    17
Oncology              13    21
Orthopedics           24    28
Pediatrics            23    31


In [64]:
#  a pivot table showing average age by blood type
pivot_age_blood_type = pd.pivot_table(
    hospital_df,
    values  = 'age',
    index   = 'blood_type',
    aggfunc = 'mean'
).round(2)

print(pivot_age_blood_type)

             age
blood_type      
A+         49.22
A-         49.43
AB+        44.36
AB-        50.00
B+         38.22
B-         24.00
O+         45.96
O-         40.42


#  Business Questions

In [49]:
## – High-cost patient profile
# Group by patient_id to calculate total cost, number of admissions, and most common diagnosis
patient_summary = hospital_df.groupby('patient_id').agg(
    total_cost=('treatment_cost_usd', 'sum'),
    num_admissions=('admission_id', 'count'),
    # Get the most common diagnosis, handling cases where there might be no diagnosis or multiple modes
    most_common_diagnosis=('diagnosis', lambda x: x.mode()[0] if not x.mode().empty else 'Unknown')
).reset_index()

# Calculate the 90th percentile for total_cost
pctile_90 = patient_summary['total_cost'].quantile(0.90)

# Filter for patients whose total cost is in the top 10%
high_cost_patients = patient_summary[patient_summary['total_cost'] >= pctile_90]

# Sort by total_cost in descending order and display the top 15 rows
high_cost_patients = high_cost_patients.sort_values(by='total_cost', ascending=False)

print("Top 10 High-Cost Patient Profiles (Top 10% of total treatment cost):")
display(high_cost_patients.head(10))

Top 10 High-Cost Patient Profiles (Top 10% of total treatment cost):


,patient_id,total_cost,num_admissions,most_common_diagnosis
91,P0111,59634.21,6,Appendicitis
22,P0026,56280.07,5,COVID-19
145,P0179,56157.75,6,Anaemia
0,P0001,51344.94,6,Typhoid
85,P0104,50613.24,6,Appendicitis
43,P0047,48736.39,6,Appendicitis
60,P0068,45654.85,4,Appendicitis
93,P0113,45081.49,4,Appendicitis
34,P0038,42969.90,6,COVID-19
55,P0063,39972.02,4,Fracture


In [50]:
#  2 – Which Department is the most efficient?
# Group by department and calculate required metrics
department_efficiency = hospital_df.groupby('department').agg(
    total_admissions=('admission_id', 'count'),
    average_length_of_stay=('length_of_stay', 'mean'),
    average_treatment_cost=('treatment_cost_usd', 'mean'),
    # Calculate percentage of admissions covered by insurance
    percent_covered_by_insurance=('insurance_covered', lambda x: ((x == 'Yes').sum() / x.size) * 100 if x.size > 0 else 0),
    # Calculate percentage of admissions that are cost outliers
    percent_cost_outliers=('cost_outlier', lambda x: (x.sum() / x.size) * 100 if x.size > 0 else 0)
).reset_index()

# Sort by average_treatment_cost descending
department_efficiency = department_efficiency.sort_values(by='average_treatment_cost', ascending=False)

# Round numerical columns for cleaner display
department_efficiency[['average_length_of_stay', 'average_treatment_cost', 'percent_covered_by_insurance', 'percent_cost_outliers']] = \
    department_efficiency[['average_length_of_stay', 'average_treatment_cost', 'percent_covered_by_insurance', 'percent_cost_outliers']].round(2)

print("Department Efficiency Report:")
display(department_efficiency)

Department Efficiency Report:


,department,total_admissions,average_length_of_stay,average_treatment_cost,percent_covered_by_insurance,percent_cost_outliers
3,Neurology,29,14.79,8298.68,55.17,0.00
6,Pediatrics,54,13.83,8241.68,50.00,0.00
0,Cardiology,42,16.79,8218.80,50.00,0.00
2,General Medicine,58,15.22,7747.42,60.34,0.00
4,Oncology,34,14.85,7673.58,32.35,0.00
1,Emergency,81,14.59,7459.86,48.15,0.00
5,Orthopedics,52,16.92,6974.65,55.77,0.00


In [42]:
# 4.3 - Which department had the highest average cost in the most recent year?

# Find the most recent year in the dataset
most_recent_year = hospital_df['admit_year'].max()
print(f"Most recent year in the data: {most_recent_year}")

# Filter data for the most recent year
df_recent_year = hospital_df[hospital_df['admit_year'] == most_recent_year]

# Calculate average cost per department for the most recent year
avg_cost_recent_year = df_recent_year.groupby('department')['treatment_cost_usd'].mean().sort_values(ascending=False).round(2)

print(f"\nAverage Treatment Cost by Department in {most_recent_year}:")
display(avg_cost_recent_year)

# Identify the department with the highest average cost
highest_cost_department = avg_cost_recent_year.index[0]
highest_average_cost = avg_cost_recent_year.iloc[0]
print(f"\nThe department with the highest average cost in {most_recent_year} was {highest_cost_department} with an average of ${highest_average_cost}.")

Most recent year in the data: 2024

Average Treatment Cost by Department in 2024:


,treatment_cost_usd
department,
Pediatrics,8649.84
Cardiology,8196.51
General Medicine,7927.45
Neurology,7744.40
Emergency,6997.80
Orthopedics,6310.83
Oncology,3602.11



The department with the highest average cost in 2024 was Pediatrics with an average of $8649.84.


In [54]:
# Challenge 3 – Yearly trend pivot
yearly_trend_pivot = pd.pivot_table(
    hospital_df,
    index=['admit_year', 'department'],
    values=['treatment_cost_usd', 'length_of_stay', 'admission_id'],
    aggfunc={'treatment_cost_usd': 'mean', 'length_of_stay': 'mean', 'admission_id': 'count'}
).round(2)

# Rename 'admission_id' column to 'admission_count' for clarity
yearly_trend_pivot = yearly_trend_pivot.rename(columns={'admission_id': 'admission_count'})

print("Multi-level Pivot Table: Mean Treatment Cost, Length of Stay, and Admission Count by Year and Department:")
display(yearly_trend_pivot)

# Answer the question: Which department had the highest average cost in the most recent year?
most_recent_year = yearly_trend_pivot.index.get_level_values('admit_year').max()

# Filter for the most recent year and find the department with the highest average treatment cost
highest_cost_department = yearly_trend_pivot.loc[most_recent_year]['treatment_cost_usd'].idxmax()
highest_average_cost = yearly_trend_pivot.loc[most_recent_year]['treatment_cost_usd'].max()

print(f"\nIn the most recent year ({int(most_recent_year)}), the department with the highest average treatment cost is '{highest_cost_department}' with an average cost of ${highest_average_cost:.2f}.")

Multi-level Pivot Table: Mean Treatment Cost, Length of Stay, and Admission Count by Year and Department:


admission_count  length_of_stay  \
admit_year department                                          
2021       Cardiology                     12           16.67   
           Emergency                      22           14.45   
           General Medicine               13           17.38   
           Neurology                       4            9.75   
           Oncology                        6           10.50   
           Orthopedics                    10           18.30   
           Pediatrics                     15           15.00   
2022       Cardiology                      7           17.71   
           Emergency                      21           14.57   
           General Medicine               11           10.82   
           Neurology                       8           16.50   
           Oncology                        7           17.14   
           Orthopedics                    15           16.60   
           Pediatrics                      9           15.67   
2023       Cardiology                      8           15.75   
           Emergency                      20           15.75   
           General Medicine               16           16.38   
           Neurology                       7           16.57   
           Oncology                       15           14.13   
           Orthopedics                    11           15.00   
           Pediatrics                     13           13.46   
2024       Cardiology                     15           17.00   
           Emergency                      18           13.50   
           General Medicine               18           15.33   
           Neurology                      10           14.20   
           Oncology                        6           18.33   
           Orthopedics                    16           17.69   
           Pediatrics                     17           12.12   

                             treatment_cost_usd  
admit_year department                            
2021       Cardiology                   7410.42  
           Emergency                    7370.48  
           General Medicine             6353.77  
           Neurology                    7876.56  
           Oncology                     8328.44  
           Orthopedics                  9286.32  
           Pediatrics                   8940.32  
2022       Cardiology                   8805.07  
           Emergency                    7995.38  
           General Medicine             9952.71  
           Neurology                    6883.35  
           Oncology                     9056.15  
           Orthopedics                  5705.13  
           Pediatrics                   9923.94  
2023       Cardiology                   8960.19  
           Emergency                    7411.74  
           General Medicine             7161.09  
           Neurology                   10949.25  
           Oncology                     8395.03  
           Orthopedics                  7569.85  
           Pediatrics                   5737.15  
2024       Cardiology                   8196.51  
           Emergency                    6997.80  
           General Medicine             7927.45  
           Neurology                    7744.40  
           Oncology                     3602.11  
           Orthopedics                  6310.83  
           Pediatrics                   8649.84


In the most recent year (2024), the department with the highest average treatment cost is 'Pediatrics' with an average cost of $8649.84.


## 📊 Hospital Data Analysis Dashboard

This section creates an interactive dashboard using Plotly to visualize key metrics and trends from the cleaned and merged hospital data. It will incorporate the insights from the business questions previously answered.

---
## 9. 🚀 Building an Interactive Dashboard <a name='s9'></a>

Now let's combine all the Plotly charts into **one interactive dashboard** using `make_subplots()`.

```python
# Same idea as Matplotlib subplots, but interactive!
fig = make_subplots(rows=2, cols=2)
fig.add_trace(chart1, row=1, col=1)
fig.add_trace(chart2, row=1, col=2)
```

In [77]:
COLORS = px.colors.qualitative.Bold

# --- Create 4x3 subplot grid ---
fig = make_subplots(
    rows=4, cols=3,
    subplot_titles=[
        'Total Treatment Cost', 'Total Admissions', 'Average Length of Stay',
        'Avg Treatment Cost by Diagnosis', 'Patient Count by Dept & Gender', 'Avg Age by Blood Type',
        'Department Avg Cost', 'Department Avg LOS', 'Top High-Cost Patients',
        'Treatment Cost vs. Length of Stay', 'Diagnosis Distribution', '' # Empty title for the last cell if not used
    ],
    specs=[
        [{'type':'indicator'}, {'type':'indicator'}, {'type':'indicator'}],
        [{'type':'xy'}, {'type':'xy'}, {'type':'xy'}],
        [{'type':'xy'}, {'type':'xy'}, {'type':'xy'}],
        [{'type':'xy'}, {'type':'domain'}, {'type':'xy'}] # Added new row for scatter and pie
    ],
    vertical_spacing=0.1,
    horizontal_spacing=0.08,
)

# ============================================================
# ROW 1 — KPI INDICATORS
# ============================================================

fig.add_trace(go.Indicator(
    mode='number',
    value=total_treatment_cost,
    number={'prefix': '$', 'valueformat': ',.0f'},
    title={'text': 'Total Treatment Cost', 'font': {'size': 14}}
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode='number',
    value=total_admissions,
    number={'valueformat': ',.0f'},
    title={'text': 'Total Admissions', 'font': {'size': 14}}
), row=1, col=2)

fig.add_trace(go.Indicator(
    mode='number',
    value=average_length_of_stay,
    number={'suffix': ' days', 'valueformat': '.1f'},
    title={'text': 'Average Length of Stay', 'font': {'size': 14}}
), row=1, col=3)

# ============================================================
# ROW 2 — CHARTS
# ============================================================

# Chart 1: Average Treatment Cost by Diagnosis
fig.add_trace(go.Bar(
    x=pivot_diagnosis_cost['treatment_cost_usd'],
    y=pivot_diagnosis_cost.index,
    orientation='h',
    marker_color=COLORS[0],
    showlegend=False
), row=2, col=1)

# Chart 2: Patient Count by Department and Gender
for i, gender in enumerate(pivot_patient_gender.columns):
    fig.add_trace(go.Bar(
        x=pivot_patient_gender.index,
        y=pivot_patient_gender[gender],
        name=gender,
        marker_color=COLORS[i+1]
    ), row=2, col=2)

# Chart 3: Average Age by Blood Type
pivot_age_blood_type = hospital_df.groupby('blood_type')['age'].mean().round(2).reset_index()
fig.add_trace(go.Bar(
    x=pivot_age_blood_type['age'],
    y=pivot_age_blood_type['blood_type'],
    orientation='h',
    marker_color=COLORS[3],
    showlegend=False
), row=2, col=3)

# ============================================================
# ROW 3 — CHARTS
# ============================================================

# Chart 4: Department Efficiency - Average Treatment Cost
fig.add_trace(go.Bar(
    x=department_efficiency['average_treatment_cost'],
    y=department_efficiency['department'],
     orientation='h',
    marker_color='#0E7C86',
    showlegend=False
), row=3, col=1)



# Chart 5: Department Efficiency - Average Length of Stay
fig.add_trace(go.Bar(
    x=department_efficiency['average_length_of_stay'],
    y=department_efficiency['department'],
    orientation='h',
    marker_color=COLORS[5],
    showlegend=False
), row=3, col=2)

# Chart 6: Top High-Cost Patients (Top 10 from previous analysis)
fig.add_trace(go.Bar(
    x=high_cost_patients['total_cost'].head(10),
    y=high_cost_patients['patient_id'].head(10),
     orientation='h',
    marker_color=COLORS[6],
    showlegend=False
), row=3, col=3)

# ============================================================
# ROW 4 — NEW CHARTS
# ============================================================

# Bar Chart: Treatment Cost vs. Length of Stay
fig.add_trace(go.Bar(
    x=hospital_df['length_of_stay'],
    y=hospital_df['treatment_cost_usd'],
    marker_color=COLORS[7],
    name='Cost vs. LOS',
    showlegend=False
), row=4, col=1)

# Pie Chart: Diagnosis Distribution
diagnosis_counts = hospital_df['diagnosis'].value_counts().reset_index()
diagnosis_counts.columns = ['diagnosis', 'count']
fig.add_trace(go.Pie(
    labels=diagnosis_counts['diagnosis'],
    values=diagnosis_counts['count'],
    name='Diagnosis Dist.',
    marker=dict(colors=COLORS),
    hole=0.3, # Optional: creates a donut chart
    showlegend=True
), row=4, col=2)

# ============================================================
# GLOBAL LAYOUT
# ============================================================
fig.update_layout(
    title=dict(
        text='<b>📊 Hospital Data Analysis Dashboard</b>',
        x=0.5, font=dict(size=20, color='#1A3A5C')
    ),
    height=1300, # Increased height to accommodate the new row
    paper_bgcolor='#F0F4F8',
    plot_bgcolor='white',
    font=dict(family='Arial', size=11),
    barmode='group',
    legend=dict(orientation='h', yanchor='bottom', y=-0.12, x=0.05),
    margin=dict(t=80, b=60, l=50, r=50)
)

fig.show()
print('\n🎉 Interactive hospital dashboard complete!')
print('   Try: hover, zoom, click legend items to filter!')


🎉 Interactive hospital dashboard complete!
   Try: hover, zoom, click legend items to filter!


### Professional INSIGHTS for the healthcare organization based on your findings:

1. Diabetes Patients Drive the Highest Treatment Costs.
The average treatment cost for Diabetes is the highest among all diagnoses, followed by Typhoid and COVID-19. This indicates that chronic disease management is consuming a significant portion of hospital resources.

2. The Orthopedics department has the highest average cost, while Cardiology also records high costs and relatively longer lengths of stay. These departments are likely major contributors to overall hospital expenditure.

3. The "Top High-Cost Patients" chart shows several patients with treatment costs exceeding 50,000, significantly above the average. The cost distribution also indicates a few extreme cases driving total expenditure.



 professional RECOMMENDATIONS for the healthcare organization based on your findings





1. Strengthen diabetes prevention and management programs.
Introduce regular patient education on diet, exercise, and medication adherence.
2. Develop outpatient diabetes clinics to reduce expensive inpatient admissions.
3. Orthopedics and Cardiology Have the Highest Department Costs

4. Optimize inventory management for implants, equipment, and specialized supplies.
Introduce standardized treatment pathways to reduce unnecessary spending.
5. Implement case management for high-cost patients.
Monitor complex cases closely to prevent avoidable complications and prolonged stays.
Use predictive analytics to identify patients at risk of becoming high-cost cases and intervene early.

NOTE
The hospital should focus on:

Reducing chronic disease costs (especially Diabetes).
Improving efficiency in high-cost departments (Orthopedics and Cardiology).
Actively managing high-cost patients through early intervention and care coordination.

